# Day 47: pandas for data analysis

UrbanKart Data Analyst Training. Put the four UrbanKart CSV files in the same folder as this notebook (in Colab: upload them with the folder icon on the left). Write your code in each empty cell and run it with Shift + Enter.

Files: UrbanKart_orders.csv, UrbanKart_products.csv, UrbanKart_customers.csv, UrbanKart_returns.csv

**Example.** Import pandas, read UrbanKart_orders.csv and show the first 5 rows.

In [1]:
import pandas as pd
orders = pd.read_csv('UrbanKart_orders.csv')
orders.head()

   order_id  order_date customer_id product_id  quantity  unit_price  discount_pct  sales       city      payment_mode order_status delivery_date
0  UK100001  2025-01-01       C0037       P002         2        2199           0.0   4398     Jaipur       Net Banking    Delivered    2025-01-03
1  UK100002  2025-01-01       C0448       P011         1         349           0.0    349  Bengaluru  Cash on Delivery    Delivered    2025-01-05
2  UK100003  2025-01-02       C0227       P002         1        2199           0.0   2199    Kolkata               UPI    Delivered    2025-01-06
3  UK100004  2025-01-02       C0406       P005         1         899           0.0    899       Pune  Cash on Delivery    Delivered    2025-01-05
4  UK100005  2025-01-03       C0106       P021         1         899           0.0    899  Bengaluru  Cash on Delivery    Delivered    2025-01-08

**T1.** How many rows and columns? Convert order_date and delivery_date to datetime and check the dtypes.

In [2]:
print(orders.shape)
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['delivery_date'] = pd.to_datetime(orders['delivery_date'])
orders.dtypes

(3288, 12)


order_id                    str
order_date       datetime64[us]
customer_id                 str
product_id                  str
quantity                  int64
unit_price                int64
discount_pct            float64
sales                     int64
city                        str
payment_mode                str
order_status                str
delivery_date    datetime64[us]

**T2.** Summary statistics of sales, and the number of missing values per column.

In [3]:
print(orders['sales'].describe().round(1))
orders.isna().sum()

count     3288.0
mean      2362.5
std       2787.6
min        103.0
25%        699.0
50%       1499.0
75%       2998.0
max      27998.0
Name: sales, dtype: float64


order_id           0
order_date         0
customer_id        0
product_id         0
quantity           0
unit_price         0
discount_pct       0
sales              0
city               0
payment_mode       0
order_status       0
delivery_date    269

**T3.** How many Delivered orders from Mumbai have sales above ₹5,000?

In [4]:
big_mumbai = orders[(orders['order_status'] == 'Delivered') & (orders['city'] == 'Mumbai') & (orders['sales'] > 5000)]
len(big_mumbai)

60

**T4.** Show the 5 biggest orders (order_id, city, sales), largest first.

In [5]:
orders.sort_values('sales', ascending=False)[['order_id', 'city', 'sales']].head(5)

      order_id       city  sales
2718  UK102719    Kolkata  27998
2650  UK102651    Kolkata  27998
2823  UK102824    Chennai  27998
1900  UK101901      Delhi  27998
2304  UK102305  Hyderabad  27998

**T5.** Add columns: year, month (YYYY-MM) and delivery_days. What is the average delivery_days?

In [6]:
orders['year'] = orders['order_date'].dt.year
orders['month'] = orders['order_date'].dt.strftime('%Y-%m')
orders['delivery_days'] = (orders['delivery_date'] - orders['order_date']).dt.days
float(round(orders['delivery_days'].mean(), 2))

4.26

**T6.** Share of orders by order_status and by payment_mode (%).

In [7]:
print((orders['order_status'].value_counts(normalize=True) * 100).round(1))
(orders['payment_mode'].value_counts(normalize=True) * 100).round(1)

order_status
Delivered    85.9
Cancelled     7.7
Returned      5.9
Shipped       0.5
Name: proportion, dtype: float64


payment_mode
UPI                 42.8
Card                21.4
Cash on Delivery    20.3
Net Banking          8.1
Wallet               7.5

**T7.** Top 5 cities by total sales, with number of orders and average order value.

In [8]:
city = orders.groupby('city').agg(sales=('sales', 'sum'), orders=('order_id', 'count'), aov=('sales', 'mean'))
city.sort_values('sales', ascending=False).head(5).round(0)

             sales  orders     aov
city                              
Mumbai     1449034     621  2333.0
Bengaluru  1257759     533  2360.0
Delhi       934227     406  2301.0
Kolkata     666461     290  2298.0
Pune        662098     269  2461.0

**T8.** Merge with products. Sales, profit (sales − quantity × cost_price) and margin % by category.

In [9]:
products = pd.read_csv('UrbanKart_products.csv')
df = orders.merge(products, on='product_id', how='left')
df['profit'] = df['sales'] - df['quantity'] * df['cost_price']
cat = df.groupby('category')[['sales', 'profit']].sum()
cat['margin_pct'] = (cat['profit'] / cat['sales'] * 100).round(1)
cat.sort_values('sales', ascending=False)

                      sales   profit  margin_pct
category                                        
Electronics         3710329  1159627        31.3
Fashion             1919658   801336        41.7
Home & Kitchen      1393850   522046        37.5
Beauty               577476   238908        41.4
Books & Stationery   166459    69138        41.5

**T9.** Pivot table: sales by category (rows) and year (columns).

In [10]:
pd.pivot_table(df, index='category', columns='year', values='sales', aggfunc='sum')

year                   2025     2026
category                            
Beauty               247509   329967
Books & Stationery    85299    81160
Electronics         1722671  1987658
Fashion              924687   994971
Home & Kitchen       664330   729520

**T10.** Monthly sales for 2026 (Jan–Sep) and the month-on-month change %.

In [11]:
m = orders[orders['year'] == 2026].groupby('month')['sales'].sum().to_frame()
m['mom_pct'] = (m['sales'].pct_change() * 100).round(1)
m

          sales  mom_pct
month                   
2026-01  352122      NaN
2026-02  383596      8.9
2026-03  489533     27.6
2026-04  417428    -14.7
2026-05  490696     17.6
2026-06  466470     -4.9
2026-07  525104     12.6
2026-08  527553      0.5
2026-09  470774    -10.8

**T11.** Merge with customers: sales by region and customer segment (pivot).

In [12]:
customers = pd.read_csv('UrbanKart_customers.csv')
df2 = orders.merge(customers[['customer_id', 'region', 'segment']], on='customer_id', how='left')
pd.pivot_table(df2, index='region', columns='segment', values='sales', aggfunc='sum', fill_value=0, margins=True)

segment      New  Premium  Regular      All
region                                     
Central    29131        0    61981    91112
East      137438   120211   408812   666461
North     477111   221313  1227936  1926360
South     618559   253136  1654964  2526659
West      446056   433732  1677392  2557180
All      1708295  1028392  5031085  7767772

**T12.** Return rate % by category (orders with status Returned ÷ all orders).

In [13]:
df['is_returned'] = df['order_status'] == 'Returned'
(df.groupby('category')['is_returned'].mean() * 100).round(1).sort_values(ascending=False)

category
Home & Kitchen        7.1
Books & Stationery    6.2
Beauty                5.9
Fashion               5.6
Electronics           5.2